<section dir="rtl"><h2>OncoPredict — مختبر تدريب IC50 وتطويره</h2><p><b>EDA · Feature Engineering · Nested Hyperparameter Tuning · Diagnostics</b></p>
<p>نسخة تطوير بحثية قابلة للتنفيذ على 83 خط خلية مخبري وثلاثة أدوية. الهدف LN_IC50 المنشور، وليس تصنيف المرضى. هذا الملف جديد؛ لا يوصف بأنه النوتبوك الذي نفذ التدريب التاريخي.</p>
<p>يشمل الكود والرسوم والنتائج المحفوظة داخل الملف نفسه. شغّل الخلايا بالترتيب أو Run All. كل تشغيل ينشئ مجلد نتائج جديدًا ويحافظ على المصادر الأصلية. اقرأ PROTOCOL_AR.md قبل تعديل التجارب.</p>
<p><b>تنبيه منهجي:</b> 218/249 قيمة خارج أعلى جرعة مختبرة؛ تطوير الخوارزمية لا يحول هذه القيم إلى قياسات موثقة داخل النطاق. الخلايا نفسها استُخدمت في التطوير سابقًا؛ هذا ليس تحققًا نهائيًا مستقلًا.</p></section>

<section dir="rtl"><h2>1. إعداد البيئة ومكان الحفظ</h2><p>اكتشاف مجلد المشروع تلقائيًا، تسجيل النسخ والبصمات، وحفظ كل تشغيل في runs. لا يُعاد تدريب النموذج القديم أو تحديث واجهة المشروع تلقائيًا.</p></section>

In [ ]:
import os,sys,json,hashlib,platform,inspect,time,uuid
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import sklearn,scipy,joblib
from scipy.stats import spearmanr
from IPython.display import display,HTML,Code
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge,MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold,GridSearchCV,ParameterGrid,learning_curve
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,confusion_matrix
from sklearn.inspection import permutation_importance
get_ipython().run_line_magic('matplotlib','inline')
parents=[Path.cwd(),*Path.cwd().parents]
PROJECT=next((p for p in parents if (p/'outputs/ic50_exploratory/cell_features_83.csv').is_file()),None)
LAB=PROJECT/'training/ic50_notebook' if PROJECT else next((p for p in parents if (p/'features.py').is_file() and (p/'data/cell_features_83.csv').is_file()),None)
if LAB is None: raise FileNotFoundError('Open the notebook inside its training folder.')
ROOT=PROJECT or LAB
sys.path.insert(0,str(LAB))
from features import FEATURES,ENGINEERED_NAMES,add_genomic_interactions
from notebook_checks import audit_cells,ranking_labels,bootstrap_gain
SOURCE=LAB/'data'  # Exact frozen copies; original study files remain read-only.
DRUGS=['Osimertinib','Erlotinib','Gefitinib']
SEEDS=[42,137,2026]
RUN=LAB/'runs'/('nested_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:6])
RUN.mkdir(parents=True);(RUN/'figures').mkdir()
COLORS=['#146a83','#c19450','#6c8591']
mpl.rcParams.update({'figure.dpi':110,'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold','axes.grid':True,'grid.alpha':.15,'savefig.facecolor':'white'})
def write_json(name,obj):
    (RUN/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
def show_fig(fig,name):
    fig.tight_layout();fig.savefig(RUN/'figures'/f'{name}.png',dpi=180,bbox_inches='tight');fig.savefig(RUN/'figures'/f'{name}.svg',bbox_inches='tight');display(fig);plt.close(fig)
def fingerprint(p):return hashlib.sha256(p.read_bytes()).hexdigest()
FILES=['cell_features_83.csv','targets_wide_83.csv','modeling_table_249.csv','fold_assignments.csv','held_out_predictions.csv','run_summary.json']
source_hashes={f:fingerprint(SOURCE/f) for f in FILES}
expected_hashes=json.loads((LAB/'data_manifest.json').read_text(encoding='utf-8'))['sha256']
assert source_hashes==expected_hashes,'Frozen training snapshot fingerprint mismatch'
frozen_catalog=json.loads((PROJECT/'outputs/research_explorer/source_manifest.json').read_text(encoding='utf-8')) if PROJECT else []
original_hashes={a['path']:a['sha256'] for a in frozen_catalog}
assert all(fingerprint(ROOT/p)==h for p,h in original_hashes.items()),'Original source fingerprint mismatch'
runtime={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,'scipy':scipy.__version__,'matplotlib':mpl.__version__}
write_json('runtime.json',runtime);write_json('source_manifest.json',source_hashes)
write_json('training_code_manifest.json',{f:fingerprint(LAB/f) for f in ['PROTOCOL_AR.md','features.py','notebook_checks.py']})
display(pd.Series(runtime,name='Version').to_frame());print('Run directory:',RUN)

<section dir="rtl"><h2>2. تحميل البيانات وتدقيق الربط</h2><p>الوحدة المستقلة هي الخلية: صف ميزات واحد وثلاثة أهداف. نرفض القيم المفقودة والازدواج، ولا نحول نقص بيانات الطفرات إلى صفر. لا تدخل معلومات الاستجابة أو أسماء الخلايا في الميزات.</p></section>

In [ ]:
cells_df=pd.read_csv(SOURCE/'cell_features_83.csv').sort_values('SangerModelID').reset_index(drop=True)
ids=cells_df.SangerModelID.to_numpy()
targets=pd.read_csv(SOURCE/'targets_wide_83.csv').set_index('SangerModelID')
assert targets.index.is_unique and set(targets.index)==set(ids)
y=targets.loc[ids,DRUGS].to_numpy(float)
x=cells_df[FEATURES].to_numpy(float)
audit_cells(ids,x,y)
assert len(ids)==83
responses=pd.read_csv(SOURCE/'modeling_table_249.csv')
assert len(responses)==249 and not responses.duplicated(['SangerModelID','DRUG_NAME']).any()
assert np.allclose(np.exp(responses.LN_IC50),responses.IC50_uM)
above=responses.IC50_uM>responses.MAX_CONC
assert int(above.sum())==218
folds=pd.read_csv(SOURCE/'fold_assignments.csv')
assert len(folds)==249 and not folds.duplicated(['SangerModelID','repeat']).any()
old_summary=json.loads((SOURCE/'run_summary.json').read_text(encoding='utf-8'))
audit=pd.DataFrame({'item':['Independent cells','Cell-drug targets','Binary inputs','Missing feature values','Missing targets','Duplicated cell-drug pairs','IC50 above maximum dose'],
                    'value':[len(ids),len(responses),len(FEATURES),int(cells_df[FEATURES].isna().sum().sum()),int(np.isnan(y).sum()),int(responses.duplicated(['SangerModelID','DRUG_NAME']).sum()),int(above.sum())]})
display(audit);display(cells_df.head());display(targets.loc[ids].head());audit.to_csv(RUN/'data_audit.csv',index=False)
fig,ax=plt.subplots(figsize=(10,3));missing=np.column_stack([np.isnan(x),np.isnan(y)]).astype(int)
ax.imshow(missing.T,aspect='auto',cmap='Greys',vmin=0,vmax=1);ax.set_yticks(range(11),FEATURES+DRUGS);ax.set_xlabel('Cell row (83 independent cells)');ax.set_title('Missingness audit: 0 = present, 1 = missing');ax.grid(False);show_fig(fig,'01_missingness')

<section dir="rtl"><h2>3. EDA: توزيع الهدف ودعم نطاق الجرعات</h2><p>نرسم LN_IC50 دون قص القيم. µM وحدة تركيز مخبرية، وليست جرعة للمريض. تجاوز القيمة المنشورة أعلى جرعة يعني محدودية دعم الاستقراء؛ لا يعني وحده تصنيفًا سريريًا للمقاومة.</p></section>

In [ ]:
display(pd.DataFrame(y,columns=DRUGS).describe().T)
fig,axes=plt.subplots(1,3,figsize=(13,3.8))
for j,(ax,drug) in enumerate(zip(axes,DRUGS)):
    ax.hist(y[:,j],bins=12,color=COLORS[j],edgecolor='white');ax.axvline(np.median(y[:,j]),color='#263d4a',linestyle='--',label='Median');ax.set(title=drug,xlabel='Published LN_IC50',ylabel='Cells');ax.legend(fontsize=8)
show_fig(fig,'02_target_histograms')
fig,ax=plt.subplots(figsize=(9,4));ax.boxplot([y[:,j] for j in range(3)],tick_labels=DRUGS,showfliers=False)
rng=np.random.default_rng(42)
for j in range(3):ax.scatter(j+1+rng.uniform(-.12,.12,len(y)),y[:,j],s=15,alpha=.55,color=COLORS[j])
ax.set(ylabel='Published LN_IC50',title='Target distributions — all 83 cells retained');show_fig(fig,'03_target_boxplots')
coverage=responses.assign(above_max=above).groupby('DRUG_NAME').above_max.agg(['sum','count']).reindex(DRUGS)
fig,ax=plt.subplots(figsize=(9,4));ax.bar(DRUGS,coverage['count']-coverage['sum'],label='Within tested dose range',color=COLORS[0]);ax.bar(DRUGS,coverage['sum'],bottom=coverage['count']-coverage['sum'],label='Above maximum tested dose',color=COLORS[1])
for i,r in enumerate(coverage.itertuples()):ax.text(i,85,f'{r.sum}/{r.count} above',ha='center',fontsize=10)
ax.set(ylim=(0,100),ylabel='Cell-drug targets',title='Published IC50 support by tested dose range');ax.legend();show_fig(fig,'04_dose_support');display(coverage)

<section dir="rtl"><h2>4. EDA: انتشار الطفرات والترابط والأنماط</h2><p>الندرة والثوابت تظهر صراحة. ارتباط ميزتين لا يثبت سببية أو أهمية بيولوجية. C797S ثابت عند صفر، لذلك ارتباطه غير معرّف؛ لا نستبدله بارتباط صفري.</p></section>

In [ ]:
feature_audit=pd.DataFrame({'feature':FEATURES,'positive_cells':x.sum(axis=0).astype(int),'prevalence':x.mean(axis=0),'unique_values':[len(np.unique(x[:,j])) for j in range(8)]})
display(feature_audit);feature_audit.to_csv(RUN/'feature_audit.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4));bars=ax.barh(FEATURES,x.sum(axis=0),color=COLORS[0]);ax.invert_yaxis();ax.bar_label(bars,padding=4);ax.set(xlim=(0,83),xlabel='Positive cells (of 83)',title='Mutation feature prevalence');show_fig(fig,'05_mutation_prevalence')
corr=cells_df[FEATURES].corr()
fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(np.ma.masked_invalid(corr.to_numpy()),cmap='RdBu_r',vmin=-1,vmax=1)
ax.set_xticks(range(8),FEATURES,rotation=75,ha='right');ax.set_yticks(range(8),FEATURES);ax.grid(False)
for a in range(8):
    for b in range(8):
        v=corr.iloc[a,b];ax.text(b,a,'N/A' if pd.isna(v) else f'{v:.2f}',ha='center',va='center',fontsize=8,color='black' if pd.isna(v) or abs(v)<.6 else 'white')
fig.colorbar(im,ax=ax,label='Binary-feature Pearson correlation');ax.set_title('Feature correlation (constant features = N/A)');show_fig(fig,'06_feature_correlation')
patterns=cells_df[FEATURES].astype(str).agg(''.join,axis=1)
pattern_counts=patterns.value_counts()
display(pattern_counts.rename('cells').to_frame());print('Distinct genomic patterns:',len(pattern_counts))
fig,ax=plt.subplots(figsize=(10,4));bars=ax.bar(pattern_counts.index,pattern_counts.values,color=COLORS[0]);ax.bar_label(bars);ax.set(xlabel='Binary pattern in declared feature order',ylabel='Cells',title='Identical inputs can have different drug responses');ax.tick_params(axis='x',rotation=45);show_fig(fig,'07_genomic_patterns')
heterogeneity=pd.DataFrame(y,columns=DRUGS).assign(pattern=patterns).groupby('pattern').agg(['count','std'])
display(heterogeneity);heterogeneity.to_csv(RUN/'within_pattern_target_variation.csv')

<section dir="rtl"><h2>5. هندسة الميزات: تفاعلات محددة قبل التدريب</h2><p>نضيف EGFR×TP53 وEGFR×KRAS وTP53×KRAS فقط. تعريفها يعتمد على X ولا يطلع على y. الاختيار والتحجيم واستبعاد الثوابت يجري داخل طيات التدريب. لا ندخل AUC أو RMSE المصدر أو الجرعة أو هوية الخلية كميزات.</p></section>

In [ ]:
display(Code(inspect.getsource(add_genomic_interactions),language='python'))
engineered=add_genomic_interactions(x)
display(pd.DataFrame(engineered,columns=ENGINEERED_NAMES).head())
fig,ax=plt.subplots(figsize=(8,3));bars=ax.bar(ENGINEERED_NAMES[-3:],engineered[:,-3:].sum(axis=0),color=COLORS[1]);ax.bar_label(bars);ax.set(ylabel='Cells with joint features',title='Predeclared genomic interactions');show_fig(fig,'08_engineered_features')

<section dir="rtl"><h2>6. التجربة وضبط المعاملات: Nested CV</h2><p>ثلاث إعادات وخمس طيات خارجية بأرقام الخلايا نفسها من الدراسة الأصلية. داخل كل تدريب خارجي أربع طيات لضبط المعاملات واختيار العائلة؛ الطية الخارجية لا تؤثر في الاختيار. جميع الأدوية للخلية تبقى معًا. النتيجة الأساسية لمسار الاختيار المتداخل، لا لأفضل عائلة نختارها بعد قراءة نتائج التقييم.</p><p>التحسين هنا ضبط معاملات وهندسة ميزات، وليس fine-tuning لنموذج أساس ضخم.</p></section>

In [ ]:
def pipeline(model,engineer=False,scale=False):
    steps=[]
    if engineer:steps.append(('engineering',FunctionTransformer(add_genomic_interactions,validate=True)))
    steps.append(('variance',VarianceThreshold()))
    if scale:steps.append(('scale',StandardScaler()))
    steps.append(('model',model))
    return Pipeline(steps)
def candidate_searches(inner):
    definitions={
      'baseline':(pipeline(DummyRegressor()),{}),
      'ridge_original':(pipeline(Ridge()),{'model__alpha':[.1,1.,10.,100.]}),
      'ridge_engineered':(pipeline(Ridge(),True,True),{'model__alpha':[.1,1.,10.,100.,1000.]}),
      'elasticnet_engineered':(pipeline(MultiTaskElasticNet(max_iter=20000,random_state=42),True,True),{'model__alpha':[.01,.1,1.],'model__l1_ratio':[.25,.75]}),
      'random_forest':(pipeline(RandomForestRegressor(n_estimators=150,random_state=42,n_jobs=1)),{'model__max_depth':[3,None],'model__min_samples_leaf':[3,8]}),
      'svr_engineered':(pipeline(MultiOutputRegressor(SVR(kernel='rbf',epsilon=.1,gamma='scale')),True,True),{'model__estimator__C':[.1,1.,10.]})}
    return {name:GridSearchCV(estimator,grid,cv=inner,scoring='neg_mean_squared_error',n_jobs=1,error_score='raise') for name,(estimator,grid) in definitions.items()}
config={'target':'LN_IC50','cells':83,'drugs':DRUGS,'features':FEATURES,'engineered_features':ENGINEERED_NAMES[-3:],'seeds':SEEDS,'outer_folds':5,'inner_folds':4,'criterion':'inner MSE only; model family selected inside each outer training fold','status':'development_internal_not_independent','development_signal_rule':'nested RMSE gain vs original Ridge >=5%; paired cell delta CI lower>0; MAE not increased'}
write_json('config.json',config)
budget=pd.DataFrame([{'candidate':k,'combinations':len(ParameterGrid(s.param_grid)),'inner_fits_per_outer_fold':4*len(ParameterGrid(s.param_grid))} for k,s in candidate_searches(4).items()])
display(budget);print('Total inner fits in 15 outer folds:',int(budget.inner_fits_per_outer_fold.sum()*15))
print('Protocol SHA256:',fingerprint(LAB/'PROTOCOL_AR.md'))

In [ ]:
t0=time.perf_counter()
names=list(candidate_searches(4))
predictions={name:np.full((3,83,3),np.nan) for name in names+['nested_selected']}
grid_rows=[];selection_rows=[];partition_rows=[];importance_rows=[];train_gap=[]
for repeat,seed in enumerate(SEEDS,1):
    assignment=folds[folds['repeat']==repeat].set_index('SangerModelID').loc[ids]
    assert assignment.seed.eq(seed).all()
    for outer_fold in range(1,6):
        test=np.flatnonzero(assignment.outer_fold.to_numpy()==outer_fold)
        train=np.flatnonzero(assignment.outer_fold.to_numpy()!=outer_fold)
        assert not set(ids[train])&set(ids[test])
        inner=list(KFold(4,shuffle=True,random_state=seed+outer_fold).split(x[train]))
        for a,b in inner:
            assert not set(ids[train[a]])&set(ids[train[b]])
        searches=candidate_searches(inner)
        for name,search in searches.items():
            search.fit(x[train],y[train])
            predictions[name][repeat-1,test]=search.predict(x[test])
            selection_rows.append({'repeat':repeat,'outer_fold':outer_fold,'candidate':name,'inner_MSE':-float(search.best_score_),'best_params':json.dumps(search.best_params_)})
            for params,score in zip(search.cv_results_['params'],search.cv_results_['mean_test_score']):
                grid_rows.append({'repeat':repeat,'outer_fold':outer_fold,'candidate':name,'params':json.dumps(params),'inner_MSE':-float(score)})
        selected=max(searches,key=lambda k:searches[k].best_score_)
        estimator=searches[selected].best_estimator_
        predictions['nested_selected'][repeat-1,test]=estimator.predict(x[test])
        selection_rows.append({'repeat':repeat,'outer_fold':outer_fold,'candidate':'nested_selected','selected_family':selected,'inner_MSE':-float(searches[selected].best_score_),'best_params':json.dumps(searches[selected].best_params_)})
        train_gap.append({'repeat':repeat,'outer_fold':outer_fold,'selected_family':selected,'training_RMSE':float(np.sqrt(mean_squared_error(y[train],estimator.predict(x[train])))),'held_out_RMSE':float(np.sqrt(mean_squared_error(y[test],estimator.predict(x[test]))))})
        imp=permutation_importance(estimator,x[test],y[test],scoring='neg_mean_squared_error',n_repeats=5,random_state=seed+outer_fold,n_jobs=1)
        importance_rows += [{'repeat':repeat,'outer_fold':outer_fold,'feature':FEATURES[j],'delta_MSE':float(imp.importances_mean[j])} for j in range(8)]
        partition_rows.append({'repeat':repeat,'outer_fold':outer_fold,'train_ids':ids[train].tolist(),'held_out_ids':ids[test].tolist(),'inner_partitions':[{'train_ids':ids[train[a]].tolist(),'validation_ids':ids[train[b]].tolist()} for a,b in inner]})
        pd.DataFrame(selection_rows).to_csv(RUN/'selections.csv',index=False)
        print(f'Repeat {repeat}/3 fold {outer_fold}/5: selected {selected}',flush=True)
assert all(np.isfinite(v).all() for v in predictions.values())
elapsed=time.perf_counter()-t0
pd.DataFrame(grid_rows).to_csv(RUN/'inner_grid_scores.csv',index=False)
pd.DataFrame(importance_rows).to_csv(RUN/'held_out_permutation_importance.csv',index=False)
pd.DataFrame(train_gap).to_csv(RUN/'train_validation_gap.csv',index=False)
write_json('partition_audit.json',partition_rows)
np.savez_compressed(RUN/'oof_predictions.npz',**predictions)
print(f'Nested CV completed in {elapsed:.1f} seconds; all predictions finite.')

<section dir="rtl"><h2>7. التحقق من إعادة إنتاج Ridge الأصلي</h2><p>نفحص أن التدريب المرجعي أعاد توقعات Ridge التاريخية نفسها؛ أي اختلاف غير متوقع يوقف الاستنتاج ولا يُعامل كتحسن. هذا ليس تحميلًا لملاءمة التدريب وتسميتها تقييمًا.</p></section>

In [ ]:
old_oof=pd.read_csv(SOURCE/'held_out_predictions.csv')
old_ridge=np.empty((3,83,3))
for repeat in range(1,4):
    q=old_oof[(old_oof.model=='ridge')&(old_oof['repeat']==repeat)]
    old_ridge[repeat-1]=q.pivot(index='SangerModelID',columns='drug',values='predicted_LN_IC50').loc[ids,DRUGS].to_numpy()
reproduction_error=float(np.max(np.abs(predictions['ridge_original']-old_ridge)))
assert np.allclose(predictions['ridge_original'],old_ridge,rtol=1e-9,atol=1e-9),f'Original Ridge mismatch: {reproduction_error}'
print('Maximum absolute original Ridge reproduction difference:',reproduction_error)

<section dir="rtl"><h2>8. المقاييس ومقارنة المرشحين</h2><p>RMSE وMAE أصغر أفضل؛ R² أكبر أفضل وقد يكون سالبًا. الأرقام من جميع توقعات التقييم عبر الإعادات، ولا تحسب من متوسط التوقعات الوصفي. مصطلح top1 هنا يخص ترتيب IC50 المنشور؛ لا يثبت العلاج الأفضل.</p></section>

In [ ]:
metrics=[];per_drug=[];oof_rows=[]
observed_rank=ranking_labels(y)
for name,p in predictions.items():
    rr=[]
    for k in range(3):
        pred_rank=ranking_labels(p[k]);valid=observed_rank!=3
        rr.append(float(np.mean((pred_rank[valid]==observed_rank[valid])&(pred_rank[valid]!=3))))
        for j,drug in enumerate(DRUGS):
            rho=float(spearmanr(y[:,j],p[k,:,j]).statistic) if np.ptp(p[k,:,j])>1e-10 else None
            per_drug.append({'candidate':name,'repeat':k+1,'drug':drug,'RMSE':float(np.sqrt(mean_squared_error(y[:,j],p[k,:,j]))),'MAE':float(mean_absolute_error(y[:,j],p[k,:,j])),'R2':float(r2_score(y[:,j],p[k,:,j])),'Spearman':rho})
            for i in range(83):oof_rows.append({'SangerModelID':ids[i],'CellLineName':cells_df.CellLineName.iloc[i],'candidate':name,'repeat':k+1,'drug':drug,'observed_LN_IC50':y[i,j],'predicted_LN_IC50':p[k,i,j],'predicted_IC50_uM':float(np.exp(p[k,i,j])),'prediction_role':'held_out_outer_fold'})
    metrics.append({'candidate':name,'RMSE':float(np.sqrt(np.mean((p-y[None,:,:])**2))),'MAE':float(np.mean(np.abs(p-y[None,:,:]))),'mean_R2_per_drug_repeat':float(np.mean([r2_score(y,p[k],multioutput='uniform_average') for k in range(3)])),'mean_top1_no_tie_accuracy':float(np.mean(rr))})
metrics_df=pd.DataFrame(metrics).set_index('candidate');per_drug_df=pd.DataFrame(per_drug)
display(metrics_df.style.format('{:.4f}'));display(per_drug_df.groupby(['candidate','drug'])[['RMSE','MAE','R2','Spearman']].mean())
metrics_df.to_csv(RUN/'metrics.csv');per_drug_df.to_csv(RUN/'per_drug_metrics.csv',index=False);pd.DataFrame(oof_rows).to_csv(RUN/'held_out_predictions.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for ax,metric in zip(axes,['RMSE','MAE']):
    vals=metrics_df[metric];bars=ax.barh(vals.index,vals,color=[COLORS[1] if n=='nested_selected' else COLORS[0] for n in vals.index]);ax.invert_yaxis();ax.bar_label(bars,fmt='%.3f',padding=4);ax.set(xlabel=f'{metric} on LN_IC50 (lower is better)',title=f'Internal OOF {metric}',xlim=(0,vals.max()*1.2))
show_fig(fig,'09_model_comparison')
selected_counts=pd.DataFrame(selection_rows).query("candidate == 'nested_selected'").selected_family.value_counts()
fig,ax=plt.subplots(figsize=(9,3));bars=ax.bar(selected_counts.index,selected_counts.values,color=COLORS[0]);ax.bar_label(bars);ax.set(ylabel='Outer folds (of 15)',title='Family selected using inner validation only');ax.tick_params(axis='x',rotation=20);show_fig(fig,'10_selection_frequency')

<section dir="rtl"><h2>9. الحقيقي مقابل المتوقع والبواقي وتحليل الأخطاء</h2><p>النقاط أدناه تستخدم متوسط LN للتوقعات الثلاث لتسهيل عرض 83 خلية، وهي وصفية. لا تُستخدم بدل مقاييس الأداء السابقة. نحافظ على الحالات ذات الخطأ الكبير؛ لا نحذفها لرفع النتيجة.</p></section>

In [ ]:
descriptive=predictions['nested_selected'].mean(axis=0)
fig,axes=plt.subplots(1,3,figsize=(13,4))
for j,(ax,drug) in enumerate(zip(axes,DRUGS)):
    ax.scatter(y[:,j],descriptive[:,j],s=28,color=COLORS[j],alpha=.75);lo=min(y[:,j].min(),descriptive[:,j].min());hi=max(y[:,j].max(),descriptive[:,j].max());ax.plot([lo,hi],[lo,hi],'--',color='#263d4a');ax.set(xlabel='Published LN_IC50',ylabel='Mean OOF predicted LN_IC50',title=drug)
show_fig(fig,'11_observed_vs_predicted')
fig,axes=plt.subplots(1,3,figsize=(13,4))
for j,(ax,drug) in enumerate(zip(axes,DRUGS)):
    ax.scatter(descriptive[:,j],y[:,j]-descriptive[:,j],s=28,color=COLORS[j],alpha=.75);ax.axhline(0,color='#263d4a',linestyle='--');ax.set(xlabel='Mean OOF predicted LN_IC50',ylabel='Residual: observed - predicted',title=drug)
show_fig(fig,'12_residuals')
errors=pd.DataFrame([{'SangerModelID':ids[i],'CellLineName':cells_df.CellLineName.iloc[i],'drug':d,'observed_ln':y[i,j],'descriptive_mean_oof_ln':descriptive[i,j],'absolute_error':abs(y[i,j]-descriptive[i,j])} for i in range(83) for j,d in enumerate(DRUGS)])
errors=errors.merge(responses[['SangerModelID','DRUG_NAME','MIN_CONC','MAX_CONC','IC50_uM']],left_on=['SangerModelID','drug'],right_on=['SangerModelID','DRUG_NAME'],validate='one_to_one')
errors['published_above_range']=errors.IC50_uM>errors.MAX_CONC
display(errors.sort_values('absolute_error',ascending=False).head(12));display(errors.groupby('published_above_range').absolute_error.agg(['count','mean','median']))
errors.to_csv(RUN/'descriptive_error_audit.csv',index=False)
gap=pd.DataFrame(train_gap)
fig,ax=plt.subplots(figsize=(9,4));ax.plot(range(1,16),gap.training_RMSE,'o-',label='Training (in-sample)',color=COLORS[2]);ax.plot(range(1,16),gap.held_out_RMSE,'s-',label='Outer held-out',color=COLORS[0]);ax.set(xlabel='Outer fold sequence',ylabel='RMSE LN_IC50',title='Training vs held-out error (diagnostic)');ax.legend();show_fig(fig,'13_train_test_gap')

<section dir="rtl"><h2>10. مصفوفة الارتباك لترتيب الدواء — وليست حساسية/مقاومة</h2><p>الصف: الدواء الذي له أدنى IC50 منشور. العمود: الدواء الذي له أدنى توقع. أربع فئات تتضمن التعادل ولا تفك التعادل عشوائيًا. كل مصفوفة تخص 83 خلية في إعادة واحدة؛ لا نجمع الإعادات كخلايا مستقلة جديدة. هذا تشخيص لترتيب مختبري؛ لا يدل على الدواء الأفضل لمريض.</p><p>غلبة دواء واحد في ترتيب القياسات قد تعطي تطابقًا مرتفعًا حتى لخط الأساس؛ لذلك نعرض توزيع الفئات ونقارن مقاييس الترتيب بخط الأساس، ولا نعرضها كنسبة ثقة أو نجاح علاجي.</p></section>

In [ ]:
labels=DRUGS+['Tie']
display(pd.Series(observed_rank).value_counts().reindex(range(4),fill_value=0).rename(index=dict(enumerate(labels))).rename('Observed lowest-IC50 cells').to_frame())
fig,axes=plt.subplots(1,3,figsize=(14,4.6));matrices=[]
for k,ax in enumerate(axes):
    cm=confusion_matrix(observed_rank,ranking_labels(predictions['nested_selected'][k]),labels=[0,1,2,3]);assert cm.sum()==83
    matrices.append(cm);ax.imshow(cm,cmap='Blues',vmin=0,vmax=83);ax.grid(False)
    ax.set_xticks(range(4),labels,rotation=45,ha='right');ax.set_yticks(range(4),labels);ax.set(xlabel='Predicted lowest IC50 drug',ylabel='Published lowest IC50 drug',title=f'Repeat {k+1} — 83 cells')
    for a in range(4):
        for b in range(4):ax.text(b,a,str(cm[a,b]),ha='center',va='center',color='white' if cm[a,b]>41 else 'black')
show_fig(fig,'14_ranking_confusion_matrices');write_json('ranking_confusion_matrices.json',{'classes':labels,'matrices':[c.tolist() for c in matrices],'n_cells_each_repeat':83,'not_sensitive_resistant':True})

<section dir="rtl"><h2>11. أهمية الميزات ومنحنى التعلم</h2><p>أهمية التبديل تحسب على خلايا التقييم الخارجي لكل طية، ولا تستخدم لاختيار الميزات في هذه الجولة. ترابط الميزات وندرتها يحدان من تفسيرها، ولا تمثل أثرًا سببيًا. الأشرطة تبين الانحراف بين الطيات، وليست فاصل ثقة.</p><p>منحنى التعلم تشخيص إضافي لـRidge محدد مسبقًا alpha=10؛ لا نختار معاملًا جديدًا بناءً على شكله ولا نفترض أن المزيد من الخلايا يضمن النجاح.</p></section>

In [ ]:
importance_df=pd.DataFrame(importance_rows)
imp_summary=importance_df.groupby('feature').delta_MSE.agg(['mean','std']).reindex(FEATURES)
display(imp_summary)
fig,ax=plt.subplots(figsize=(10,4));ax.barh(imp_summary.index,imp_summary['mean'],xerr=imp_summary['std'],color=COLORS[0],capsize=3);ax.axvline(0,color='#263d4a');ax.invert_yaxis();ax.set(xlabel='Increase in held-out MSE after permutation',title='Raw-input permutation importance: mean ± fold SD');show_fig(fig,'15_feature_importance')
sizes,train_scores,val_scores=learning_curve(pipeline(Ridge(alpha=10),True,True),x,y,cv=KFold(5,shuffle=True,random_state=42),train_sizes=[.4,.7,1.],scoring='neg_mean_squared_error',shuffle=True,random_state=42,n_jobs=1,error_score='raise')
train_rmse=np.sqrt(-train_scores);val_rmse=np.sqrt(-val_scores)
fig,ax=plt.subplots(figsize=(8,4));ax.errorbar(sizes,train_rmse.mean(axis=1),yerr=train_rmse.std(axis=1),marker='o',label='Training',color=COLORS[2],capsize=4);ax.errorbar(sizes,val_rmse.mean(axis=1),yerr=val_rmse.std(axis=1),marker='s',label='Validation',color=COLORS[0],capsize=4);ax.set(xlabel='Training cells per fold',ylabel='RMSE LN_IC50',title='Fixed Ridge alpha=10: diagnostic learning curve');ax.legend();show_fig(fig,'16_learning_curve')
pd.DataFrame({'training_cells':sizes,'training_RMSE':train_rmse.mean(axis=1),'validation_RMSE':val_rmse.mean(axis=1),'validation_fold_SD':val_rmse.std(axis=1)}).to_csv(RUN/'learning_curve.csv',index=False)

<section dir="rtl"><h2>12. فاصل فرق الأداء وقرار التجربة</h2><p>إعادة أخذ عينات بالخلايا تبقي الأدوية والإعادات معًا. الفاصل مشروط بالتوقعات المحفوظة ولا يشمل إعادة تدريب أو عدم يقين التوسع إلى خلايا جديدة. معيار الإشارة المسبق: تحسن RMSE ≥5% فوق Ridge الأصلي، وفاصل الفرق موجب، وMAE لا يزيد. لا يساوي اجتيازه اعتمادًا.</p></section>

In [ ]:
comparison=bootstrap_gain(y,old_ridge,predictions['nested_selected'])
mae_ok=metrics_df.loc['nested_selected','MAE']<=metrics_df.loc['ridge_original','MAE']
signal=bool(comparison['relative_gain']>=.05 and comparison['conditional_95_interval'][0]>0 and mae_ok)
comparison['MAE_not_increased']=bool(mae_ok);comparison['predeclared_development_signal']=signal
display(pd.Series(comparison).to_frame('Result'))
fig,ax=plt.subplots(figsize=(9,2.8));lo,hi=comparison['conditional_95_interval'];ax.plot([lo,hi],[0,0],color=COLORS[0],linewidth=4);ax.scatter([comparison['delta_RMSE']],[0],color=COLORS[0],s=70);ax.axvline(0,color='#263d4a',linestyle='--');ax.set(xlabel='RMSE(original Ridge) - RMSE(nested selection); positive favors development',yticks=[],title='Conditional paired-cell 95% interval');show_fig(fig,'17_paired_improvement_interval')
decision='ظهرت إشارة تطوير داخلية وفق القاعدة المسبقة؛ تحتاج تحققًا جديدًا مستقلًا قبل أي اعتماد.' if signal else 'لم تتحقق إشارة التطوير المسبقة؛ نحفظ النتائج كاملة ولا نعتمد نسخة محسّنة بناءً عليها.'
display(HTML('<div dir="rtl"><h3>قرار الجولة</h3><p>'+decision+'</p><p>محدودية IC50 خارج نطاق الجرعات باقية، ولا يثبت هذا الاختبار اختيار علاج لمريض.</p></div>'))
write_json('comparison_vs_original_ridge.json',comparison)

<section dir="rtl"><h2>13. ملاءمة نموذج تطويري كامل وحفظه — ليست تقييمًا</h2><p>نختار العائلة والمعاملات عبر تحقق داخلي جديد على البيانات كاملة ثم نلائمها لحفظ نموذج قابل لإعادة الاستخدام البحثي. هذا نموذج مرشح، ولا نعلن نجاحه من خطأ التدريب. يظل النموذج التاريخي وواجهة المشروع دون تعديل.</p></section>

In [ ]:
full_inner=list(KFold(5,shuffle=True,random_state=31415).split(x))
full_searches=candidate_searches(full_inner)
for search in full_searches.values():search.fit(x,y)
full_family=max(full_searches,key=lambda k:full_searches[k].best_score_)
full_pipeline=full_searches[full_family].best_estimator_
artifact={'model':full_pipeline,'features':FEATURES,'drugs':DRUGS,'target':'LN_IC50','unit_conversion':'IC50_uM = exp(LN_IC50)','status':'development_candidate_not_approved','n_training_cells':83,'C797S_supported':False,'source_hashes':source_hashes,'selected_family':full_family,'selected_params':full_searches[full_family].best_params_}
joblib.dump(artifact,RUN/'development_model.joblib')
reloaded=joblib.load(RUN/'development_model.joblib')  # Our own newly-created artifact only.
assert np.allclose(reloaded['model'].predict(x),full_pipeline.predict(x))
refit=full_pipeline.predict(x)
pd.DataFrame([{'SangerModelID':ids[i],'drug':d,'predicted_LN_IC50':refit[i,j],'predicted_IC50_uM':float(np.exp(refit[i,j])),'prediction_role':'in_sample_development_refit_not_validation'} for i in range(83) for j,d in enumerate(DRUGS)]).to_csv(RUN/'refit_predictions_not_validation.csv',index=False)
assert all(fingerprint(ROOT/p)==h for p,h in original_hashes.items()),'Original sources changed during execution'
summary={'completed_at_utc':datetime.now(timezone.utc).isoformat(),'status':'development_experiment_completed_not_approved','n_cells':83,'n_cell_drug_pairs':249,'nested_cv_runtime_seconds':elapsed,'original_Ridge_reproduction_max_abs':reproduction_error,'metrics':metrics_df.reset_index().to_dict(orient='records'),'comparison':comparison,'decision_ar':decision,'full_refit_family':full_family,'full_refit_params':full_searches[full_family].best_params_,'original_sources_verified_unchanged':len(original_hashes),'training_snapshots_verified_unchanged':source_hashes==expected_hashes,'figures':len(list((RUN/'figures').glob('*.png')))}
write_json('run_summary.json',summary)
(LAB/'latest_run.txt').write_text(RUN.name,encoding='utf-8')
(RUN/'RESULTS_AR.md').write_text('# نتائج جولة تطوير IC50\n\n'+decision+'\n\nRMSE مسار الاختيار: '+str(metrics_df.loc['nested_selected','RMSE'])+'؛ Ridge الأصلي: '+str(metrics_df.loc['ridge_original','RMSE'])+'\n\nهذه نتائج تطوير داخلي؛ لا تحديث تلقائي للنموذج السابق أو واجهة البرنامج.\n',encoding='utf-8')
print('Saved candidate family:',full_family,'parameters:',artifact['selected_params'])
print('All original sources unchanged. Outputs:',RUN)
display(HTML('<div dir="rtl"><h3>اكتمل التشغيل</h3><p>الحزمة تضم التوقعات المحجوبة، جميع نتائج البحث، التقسيمات، 17 رسمًا، تدقيق الأخطاء، والنموذج المرشح. لا تحذف النتائج غير الناجحة أو تعيد البحث دون بروتوكول نسخة جديدة.</p></div>'))

<section dir="rtl"><h2>14. المراجع وماذا نطور لاحقًا</h2><p>نبدأ التطوير اللاحق من دليل هذه الجولة: ندرة الطفرات، تكرار أنماط المدخلات، دعم القياس، ومقدار تحسن التقييم المحجوب. لا نوسع الشبكة لمجرد أن النتيجة لم تتحسن. أي نسخة لاحقة تحفظ بروتوكولها وإعداداتها ونتيجتها منفصلة.</p>
<ul><li><a href="https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html">scikit-learn — Nested versus non-nested cross-validation</a></li><li><a href="https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html">scikit-learn — permutation importance and correlated features</a></li><li>ملفات الدراسة الأصلية في outputs/ic50_exploratory، والبروتوكول المحلي PROTOCOL_AR.md.</li></ul>
<p><b>لا يوجد تدريب Sensitive/Resistant أو توصية علاج في هذه الجولة.</b> مصفوفة الترتيب منفصلة عن مقاييس الانحدار ولا تغيّر الهدف العلمي.</p></section>